# Working with Files in Python
a program's variables disappear when it stops ; FILES keep data permanently (persistence).

    1. paths          ---> where is the file ?   (pathlib)
    2. text files     ---> open / read / write / append , encoding
    3. CSV and JSON   ---> tables and structured data
    4. binary files   ---> images , models , bytes , hashing , pickle
    5. big files      ---> streaming , chunks
    6. safe writing   ---> atomic writes , temporary folders

all examples work inside a demo folder  `files_demo/`  that this notebook creates.

In [1]:
# setup : start every run with a clean demo folder (reproducible notebook)
from pathlib import Path
import shutil

BASE = Path("files_demo")
if BASE.exists():
    shutil.rmtree(BASE)
BASE.mkdir()
print("working folder:", BASE.resolve().name, "| exists:", BASE.exists())

working folder: files_demo | exists: True


## 1. Paths with pathlib

In [2]:
# build paths with the  /  operator (works on Windows , Linux and macOS)
p = BASE / "data" / "raw" / "sales_2026.csv"
print(p)
print("name   :", p.name)          # file name with extension
print("stem   :", p.stem)          # file name without extension
print("suffix :", p.suffix)        # extension
print("parent :", p.parent)        # the folder that contains it
print("parts  :", p.parts)
print("exists :", p.exists())

files_demo/data/raw/sales_2026.csv
name   : sales_2026.csv
stem   : sales_2026
suffix : .csv
parent : files_demo/data/raw
parts  : ('files_demo', 'data', 'raw', 'sales_2026.csv')
exists : False


In [3]:
# relative vs absolute paths
print("current working directory:", Path.cwd().name)
print("relative:", p)
print("absolute:", p.resolve().is_absolute())
print("with a new suffix:", p.with_suffix(".parquet").name)
print("with a new name  :", p.with_name("sales_2025.csv").name)

current working directory: nbrun
relative: files_demo/data/raw/sales_2026.csv
absolute: True
with a new suffix: sales_2026.parquet
with a new name  : sales_2025.csv


In [4]:
# create folders : parents=True creates the whole chain , exist_ok=True = no error if it exists
(BASE / "data" / "raw").mkdir(parents=True, exist_ok=True)
(BASE / "data" / "processed").mkdir(parents=True, exist_ok=True)
(BASE / "models").mkdir(exist_ok=True)
print(sorted(str(d.relative_to(BASE)) for d in BASE.rglob("*") if d.is_dir()))

['data', 'data/processed', 'data/raw', 'models']


In [5]:
# old style (os.path) ---> you will see it in older code
import os
old = os.path.join("files_demo", "data", "raw", "sales_2026.csv")
print(old)
print(os.path.basename(old), "|", os.path.splitext(old)[1], "|", os.path.exists(old))

files_demo/data/raw/sales_2026.csv
sales_2026.csv | .csv | False


## 2. Text files : open , read , write

| mode | meaning | if file exists | if file does NOT exist |
|---|---|---|---|
| `"r"` | read (default) | read it | FileNotFoundError |
| `"w"` | write | **ERASE** content | create |
| `"a"` | append | add at the end | create |
| `"x"` | exclusive create | FileExistsError | create |
| `"b"` | binary (with r/w/a/x) | `"rb"` , `"wb"` | |
| `"+"` | read AND write | `"r+"` | |

In [6]:
notes = BASE / "notes.txt"

# write : "w" creates the file or ERASES the old content
with open(notes, "w", encoding="utf-8") as f:
    f.write("first line\n")
    f.write("second line\n")
    f.writelines(["third line\n", "fourth line\n"])
print("written bytes:", notes.stat().st_size)

written bytes: 46


In [7]:
# read the whole file as ONE string
with open(notes, encoding="utf-8") as f:
    content = f.read()
print(repr(content))

'first line\nsecond line\nthird line\nfourth line\n'


In [8]:
# read line by line (memory friendly : one line at a time)
with open(notes, encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        print(i, repr(line), "--->", line.rstrip("\n"))

1 'first line\n' ---> first line
2 'second line\n' ---> second line
3 'third line\n' ---> third line
4 'fourth line\n' ---> fourth line


In [9]:
# readline() reads ONE line , readlines() reads ALL lines into a list
with open(notes, encoding="utf-8") as f:
    print(repr(f.readline()))
    print(repr(f.readline()))
    print(f.readlines())          # the REST of the file (the file position moved)

'first line\n'
'second line\n'
['third line\n', 'fourth line\n']


In [10]:
# "a" append : add to the end without erasing
with open(notes, "a", encoding="utf-8") as f:
    print("fifth line (written with print)", file=f)
print(notes.read_text(encoding="utf-8").splitlines())

['first line', 'second line', 'third line', 'fourth line', 'fifth line (written with print)']


In [11]:
# "w" ERASES everything ---> the most common way to lose data !
backup = BASE / "important.txt"
backup.write_text("10 years of results\n", encoding="utf-8")
with open(backup, "w", encoding="utf-8") as f:
    pass                                    # opened with "w" ---> already empty
print("size after opening with 'w':", backup.stat().st_size)

# "x" protects you : it refuses to overwrite
try:
    with open(backup, "x", encoding="utf-8") as f:
        f.write("new")
except FileExistsError as e:
    print("FileExistsError:", e.filename)

size after opening with 'w': 0
FileExistsError: files_demo/important.txt


In [12]:
# pathlib shortcuts for small files
cfg = BASE / "hello.txt"
cfg.write_text("سلام دنیا\nHello world\n", encoding="utf-8")
print(cfg.read_text(encoding="utf-8"))

سلام دنیا
Hello world



In [13]:
# the file position : tell() and seek()
with open(notes, encoding="utf-8") as f:
    print("position:", f.tell())
    f.readline()
    print("position after one line:", f.tell())
    f.seek(0)                               # go back to the start
    print("first line again:", f.readline().strip())

position: 0
position after one line: 11
first line again: first line


## 3. Encoding : text <---> bytes  (very important for Persian text)

In [14]:
fa = BASE / "persian.txt"
fa.write_text("قیمت: ۱۲۵۰۰ تومان", encoding="utf-8")
raw = fa.read_bytes()
print("characters:", len(fa.read_text(encoding="utf-8")))
print("bytes     :", len(raw))
print("first bytes:", raw[:8])

characters: 17
bytes     : 31
first bytes: b'\xd9\x82\xdb\x8c\xd9\x85\xd8\xaa'


In [15]:
# reading with the WRONG encoding
try:
    fa.read_text(encoding="ascii")
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e.reason, "| byte position", e.start)

print("latin-1 (no error but garbage):", fa.read_text(encoding="latin-1")[:12])
print("errors='replace'              :", fa.read_text(encoding="ascii", errors="replace")[:12])

UnicodeDecodeError: ordinal not in range(128) | byte position 0
latin-1 (no error but garbage): ÙÛÙØª: Û±
errors='replace'              : ��������: ��


In [16]:
# utf-8-sig : UTF-8 with a BOM (Byte Order Mark) ---> Excel opens Persian CSV correctly
excel_csv = BASE / "for_excel.csv"
excel_csv.write_text("نام,شهر\nمریم,تهران\n", encoding="utf-8-sig")
print(excel_csv.read_bytes()[:3])                       # the BOM bytes
print(repr(excel_csv.read_text(encoding="utf-8")[:4]))  # BOM appears as '\ufeff'
print(repr(excel_csv.read_text(encoding="utf-8-sig")[:4]))

b'\xef\xbb\xbf'
'\ufeffنام'
'نام,'


## 4. Listing and finding files

In [17]:
# create some demo files
for name in ["epoch_1.pt", "epoch_2.pt", "epoch_10.pt", "epoch_9.pt", "config.json", "train.log"]:
    (BASE / "models" / name).write_text("demo", encoding="utf-8")

models = BASE / "models"
print("iterdir:", sorted(p.name for p in models.iterdir()))
print("glob *.pt :", sorted(p.name for p in models.glob("*.pt")))
print("rglob *.txt (all sub-folders):", sorted(str(p.relative_to(BASE)) for p in BASE.rglob("*.txt")))

iterdir: ['config.json', 'epoch_1.pt', 'epoch_10.pt', 'epoch_2.pt', 'epoch_9.pt', 'train.log']
glob *.pt : ['epoch_1.pt', 'epoch_10.pt', 'epoch_2.pt', 'epoch_9.pt']
rglob *.txt (all sub-folders): ['hello.txt', 'important.txt', 'notes.txt', 'persian.txt']


In [18]:
# sorting file names : alphabetical order is NOT numeric order
checkpoints = list(models.glob("epoch_*.pt"))
print("alphabetical:", sorted(p.name for p in checkpoints))
latest_wrong = sorted(checkpoints)[-1]

def epoch_number(path):
    return int(path.stem.split("_")[1])

latest = max(checkpoints, key=epoch_number)
print("latest (wrong):", latest_wrong.name)
print("latest (right):", latest.name)

alphabetical: ['epoch_1.pt', 'epoch_10.pt', 'epoch_2.pt', 'epoch_9.pt']
latest (wrong): epoch_9.pt
latest (right): epoch_10.pt


In [19]:
# file information
info = (models / "config.json").stat()
print("size (bytes):", info.st_size)
import datetime
print("modified    :", datetime.datetime.fromtimestamp(info.st_mtime).strftime("%Y-%m-%d"))
print("is_file / is_dir:", (models / "config.json").is_file(), models.is_dir())

size (bytes): 4
modified    : 2026-11-07
is_file / is_dir: True True


## 5. CSV files (tables)

In [20]:
import csv
rows = [
    {"id": 1, "name": "Ali", "city": "Tehran", "income": 52000},
    {"id": 2, "name": "Sara", "city": "Shiraz", "income": 61000},
    {"id": 3, "name": "Reza, Jr.", "city": "Tabriz", "income": 48000},   # a comma INSIDE a value
]
customers = BASE / "data" / "raw" / "customers.csv"
with open(customers, "w", encoding="utf-8", newline="") as f:     # newline="" for csv !
    writer = csv.DictWriter(f, fieldnames=["id", "name", "city", "income"])
    writer.writeheader()
    writer.writerows(rows)
print(customers.read_text(encoding="utf-8"))

id,name,city,income
1,Ali,Tehran,52000
2,Sara,Shiraz,61000
3,"Reza, Jr.",Tabriz,48000



In [21]:
# read with DictReader : each row becomes a dict ; ALL values are STRINGS
with open(customers, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    data = list(reader)
print(data[2])
print(type(data[0]["income"]))

incomes = [int(r["income"]) for r in data]          # convert types yourself
print("mean income:", sum(incomes) / len(incomes))

{'id': '3', 'name': 'Reza, Jr.', 'city': 'Tabriz', 'income': '48000'}
<class 'str'>
mean income: 53666.666666666664


In [22]:
# why use the csv module instead of split(",") ?
line = customers.read_text(encoding="utf-8").splitlines()[3]
print("split(','):", line.split(","))                  # WRONG : 5 fields
print("csv module:", next(csv.reader([line])))         # RIGHT : 4 fields

split(','): ['3', '"Reza', ' Jr."', 'Tabriz', '48000']
csv module: ['3', 'Reza, Jr.', 'Tabriz', '48000']


In [23]:
# csv.reader / csv.writer : rows as lists , other delimiters (e.g. ;  or  tab)
tsv = BASE / "data" / "raw" / "scores.tsv"
with open(tsv, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f, delimiter="\t")
    w.writerow(["model", "accuracy"])
    w.writerows([["rf", 0.912], ["xgb", 0.925]])
with open(tsv, encoding="utf-8", newline="") as f:
    for row in csv.reader(f, delimiter="\t"):
        print(row)

['model', 'accuracy']
['rf', '0.912']
['xgb', '0.925']


In [24]:
# the same file with pandas (the usual tool for data analysis)
import pandas as pd
df = pd.read_csv(customers)
print(df)
print(df.dtypes.to_dict())            # pandas converts types automatically
df.to_csv(BASE / "data" / "processed" / "customers_clean.csv", index=False)

   id       name    city  income
0   1        Ali  Tehran   52000
1   2       Sara  Shiraz   61000
2   3  Reza, Jr.  Tabriz   48000
{'id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'city': <StringDtype(storage='python', na_value=nan)>, 'income': dtype('int64')}


## 6. JSON and JSON Lines

In [25]:
import json
config = {
    "model": {"name": "vit", "layers": 12},
    "train": {"lr": 3e-4, "epochs": 30, "augment": True},
    "labels": ["گربه", "سگ", "پرنده"],
}
cfg_path = BASE / "config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(cfg_path.read_text(encoding="utf-8"))

{
  "model": {
    "name": "vit",
    "layers": 12
  },
  "train": {
    "lr": 0.0003,
    "epochs": 30,
    "augment": true
  },
  "labels": [
    "گربه",
    "سگ",
    "پرنده"
  ]
}


In [26]:
with open(cfg_path, encoding="utf-8") as f:
    loaded = json.load(f)
print(loaded == config)
print(loaded["train"]["lr"], loaded["labels"][0])

# dumps / loads work with STRINGS , dump / load work with FILES
s = json.dumps({"a": 1})
print(type(s), json.loads(s))

True
0.0003 گربه
<class 'str'> {'a': 1}


In [27]:
# JSON Lines (.jsonl) : ONE JSON object per line ---> streaming , appending , huge datasets
examples = [
    {"messages": [{"role": "user", "content": "2+2?"}, {"role": "assistant", "content": "4"}]},
    {"messages": [{"role": "user", "content": "capital of Iran?"}, {"role": "assistant", "content": "Tehran"}]},
]
jl = BASE / "data" / "train.jsonl"
with open(jl, "w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with open(jl, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        record = json.loads(line)
        print(line_no, record["messages"][0]["content"], "--->", record["messages"][1]["content"])

1 2+2? ---> 4
2 capital of Iran? ---> Tehran


## 7. Binary files , hashing and pickle

In [28]:
# write and read raw bytes : modes "wb" and "rb"
blob = BASE / "image.png"
png_signature = b"\x89PNG\r\n\x1a\n"           # the first 8 bytes of every PNG file
blob.write_bytes(png_signature + bytes(range(256)) * 40)

with open(blob, "rb") as f:
    header = f.read(8)
print("size:", blob.stat().st_size, "bytes")
print("header:", header, "| is PNG:", header == png_signature)

size: 10248 bytes
header: b'\x89PNG\r\n\x1a\n' | is PNG: True


In [29]:
# detect the REAL type of a file from its 'magic number' (do not trust the extension)
MAGIC = {b"\x89PNG": "png", b"\xff\xd8\xff": "jpeg", b"PK\x03\x04": "zip", b"%PDF": "pdf"}

def detect_type(path):
    head = Path(path).read_bytes()[:8]
    for magic, kind in MAGIC.items():
        if head.startswith(magic):
            return kind
    return "unknown"

fake = BASE / "photo.jpg"
fake.write_bytes(png_signature + b"...")             # extension says jpg , content is png
print(detect_type(blob), detect_type(fake), detect_type(cfg_path))

png png unknown


In [30]:
# hash a file in CHUNKS ---> check integrity of a downloaded model / dataset
import hashlib

def sha256_of(path, chunk_size=8192):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

digest = sha256_of(blob)
print(digest[:16], "...")
print("same file , same hash :", sha256_of(blob) == digest)
blob.write_bytes(blob.read_bytes()[:-1] + b"X")      # change ONE byte
print("one byte changed      :", sha256_of(blob) == digest)

65c23cac92f7e4fb ...
same file , same hash : True
one byte changed      : False


In [31]:
# pickle : save ANY python object (dict , list , trained model ...)
import pickle
state = {"weights": [0.12, -0.5, 1.3], "bias": 0.05, "classes": ("cat", "dog")}
pk = BASE / "models" / "model.pkl"
with open(pk, "wb") as f:
    pickle.dump(state, f)
with open(pk, "rb") as f:
    restored = pickle.load(f)
print(restored, restored == state)
# SECURITY : never unpickle a file you do not trust ---> it can run any code on your computer

{'weights': [0.12, -0.5, 1.3], 'bias': 0.05, 'classes': ('cat', 'dog')} True


## 8. Large files : stream , don't load everything

In [32]:
# create a 'large' log file (200,000 lines)
import random
random.seed(0)
log = BASE / "server.log"
levels = ["INFO"] * 90 + ["WARNING"] * 8 + ["ERROR"] * 2
modules = ["auth", "db", "payment", "model"]
with open(log, "w", encoding="utf-8") as f:
    for i in range(200_000):
        t = i // 4                                     # 4 requests per second
        f.write(f"2026-11-07 {10 + t // 3600:02d}:{t // 60 % 60:02d}:{t % 60:02d} {random.choice(levels)} "
                f"{random.choice(modules)} request {i}\n")
print(f"{log.stat().st_size / 1e6:.1f} MB")

9.0 MB


In [33]:
# stream line by line : memory stays small , whatever the file size
from collections import Counter
errors_by_module = Counter()
n_lines = 0
with open(log, encoding="utf-8") as f:
    for line in f:
        n_lines += 1
        date, time_, level, module, _ = line.split(maxsplit=4)
        if level == "ERROR":
            errors_by_module[module] += 1
print("lines:", n_lines)
print("errors:", errors_by_module.most_common())

lines: 200000
errors: [('payment', 1022), ('auth', 1016), ('model', 976), ('db', 973)]


In [34]:
# a generator that yields only the lines we need (reusable , lazy)
def read_lines(path, contains=None):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if contains is None or contains in line:
                yield line.rstrip("\n")

first_three_payment_errors = []
for line in read_lines(log, contains="ERROR payment"):
    first_three_payment_errors.append(line)
    if len(first_three_payment_errors) == 3:
        break                               # stops reading the file early
print(*first_three_payment_errors, sep="\n")

2026-11-07 10:01:30 ERROR payment request 362
2026-11-07 10:01:49 ERROR payment request 439
2026-11-07 10:03:46 ERROR payment request 906


In [35]:
# the LAST n lines (like  tail -n 3) with a deque
from collections import deque
with open(log, encoding="utf-8") as f:
    last = deque(f, maxlen=3)
print("".join(last), end="")

2026-11-07 23:53:19 INFO auth request 199997
2026-11-07 23:53:19 INFO auth request 199998
2026-11-07 23:53:19 INFO payment request 199999


## 9. Safe writing : atomic writes and temporary folders

In [36]:
# atomic write : write to a TEMP file , then replace in ONE step
# if the program crashes in the middle , the old file is still complete (no half-written checkpoint)
import os, tempfile

def atomic_write_text(path, text, encoding="utf-8"):
    path = Path(path)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=path.name, suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding=encoding) as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())              # make sure the bytes are on the disk
        os.replace(tmp, path)                  # atomic on the same file system
    except BaseException:
        Path(tmp).unlink(missing_ok=True)      # clean up the temp file on ANY error
        raise

atomic_write_text(BASE / "models" / "best.json", json.dumps({"epoch": 7, "val_loss": 0.21}))
print((BASE / "models" / "best.json").read_text())
print("temp files left:", list((BASE / "models").glob("*.tmp")))

{"epoch": 7, "val_loss": 0.21}
temp files left: []


In [37]:
# temporary folder : deleted automatically at the end of the with-block
with tempfile.TemporaryDirectory() as tmp:
    t = Path(tmp) / "scratch.txt"
    t.write_text("temporary", encoding="utf-8")
    print("inside :", t.exists())
print("after  :", t.exists())

inside : True
after  : False


In [38]:
# copy , move , delete
src = BASE / "notes.txt"
shutil.copy(src, BASE / "notes_copy.txt")
shutil.move(BASE / "notes_copy.txt", BASE / "data" / "notes_moved.txt")
(BASE / "data" / "notes_moved.txt").unlink()          # delete a file
print(sorted(p.name for p in BASE.iterdir() if p.is_file()))

['config.json', 'for_excel.csv', 'hello.txt', 'image.png', 'important.txt', 'notes.txt', 'persian.txt', 'photo.jpg', 'server.log']


## 10. Common file errors

In [39]:
tests = [
    ("read a missing file", lambda: open(BASE / "missing.csv", encoding="utf-8")),
    ("open a folder as a file", lambda: open(BASE / "data", encoding="utf-8")),
    ("mkdir that exists", lambda: (BASE / "data").mkdir()),
    ("read binary as utf-8", lambda: (BASE / "image.png").read_text(encoding="utf-8")),
    ("write to a closed file", lambda: (lambda f: (f.close(), f.write("x")))(open(BASE / "c.txt", "w"))),
]
for label, action in tests:
    try:
        action()
    except (OSError, ValueError) as e:
        print(f"{label:<26} -> {type(e).__name__}")

read a missing file        -> FileNotFoundError
open a folder as a file    -> IsADirectoryError
mkdir that exists          -> FileExistsError
read binary as utf-8       -> UnicodeDecodeError
write to a closed file     -> ValueError


## 11. Applications in AI , Machine Learning and Expert Systems

In [40]:
# Application 1 : an image dataset stored as folders  (one folder per class , like torchvision ImageFolder)
dataset = BASE / "dataset"
counts = {"cat": 6, "dog": 5, "bird": 3}
for label, n in counts.items():
    (dataset / label).mkdir(parents=True, exist_ok=True)
    for i in range(n):
        (dataset / label / f"{label}_{i:03d}.jpg").write_bytes(b"\xff\xd8\xff fake image")

classes = sorted(d.name for d in dataset.iterdir() if d.is_dir())     # sorted ---> stable ids
label2id = {c: i for i, c in enumerate(classes)}
samples = [(p, label2id[p.parent.name]) for p in sorted(dataset.rglob("*.jpg"))]
print("classes :", label2id)
print("samples :", len(samples), "| first:", samples[0][0].name, samples[0][1])
print("per class:", Counter(classes[y] for _, y in samples))

classes : {'bird': 0, 'cat': 1, 'dog': 2}
samples : 14 | first: bird_000.jpg 0
per class: Counter({'cat': 6, 'dog': 5, 'bird': 3})


In [41]:
# Application 2 : deterministic train/val split saved as a MANIFEST csv (reproducible experiments)
random.seed(42)
shuffled = samples[:]
random.shuffle(shuffled)
n_val = int(len(shuffled) * 0.25)
manifest = BASE / "data" / "manifest.csv"
with open(manifest, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["path", "label", "split"])
    for i, (path, y) in enumerate(shuffled):
        w.writerow([path.relative_to(BASE).as_posix(), y, "val" if i < n_val else "train"])

m = pd.read_csv(manifest)
print(m.head(4).to_string(index=False))
print(m["split"].value_counts().to_dict())

                   path  label split
dataset/cat/cat_005.jpg      1   val
dataset/dog/dog_003.jpg      2   val
dataset/cat/cat_004.jpg      1   val
dataset/cat/cat_003.jpg      1 train
{'train': 11, 'val': 3}


In [42]:
# Application 3 : experiment logging ---> append one CSV row per epoch , read it back later
metrics_file = BASE / "metrics.csv"
history = [(1, 0.92, 0.88), (2, 0.61, 0.70), (3, 0.45, 0.62), (4, 0.38, 0.64), (5, 0.33, 0.66)]
for epoch, train_loss, val_loss in history:
    new_file = not metrics_file.exists()
    with open(metrics_file, "a", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["epoch", "train_loss", "val_loss"])
        w.writerow([epoch, train_loss, val_loss])

log_df = pd.read_csv(metrics_file)
best = log_df.loc[log_df["val_loss"].idxmin()]
print(log_df.to_string(index=False))
print(f"best epoch = {int(best.epoch)} (val_loss = {best.val_loss})")

 epoch  train_loss  val_loss
     1        0.92      0.88
     2        0.61      0.70
     3        0.45      0.62
     4        0.38      0.64
     5        0.33      0.66
best epoch = 3 (val_loss = 0.62)


In [43]:
# Application 4 : validate a JSONL fine-tuning dataset BEFORE uploading it (skip and report bad lines)
jl_bad = BASE / "data" / "train_raw.jsonl"
jl_bad.write_text(
    '{"messages": [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}]}\n'
    '{"messages": [{"role": "user", "content": "no answer"}]}\n'
    '{"messages": [{"role": "user", "content": "broken"\n'
    '\n'
    '{"messages": [{"role": "user", "content": "2+2"}, {"role": "assistant", "content": "4"}]}\n',
    encoding="utf-8",
)

def validate_example(obj):
    msgs = obj.get("messages")
    if not isinstance(msgs, list) or not msgs:
        raise ValueError("missing 'messages' list")
    if msgs[-1].get("role") != "assistant":
        raise ValueError("last message must be from the assistant")

good, problems = [], []
with open(jl_bad, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            problems.append((line_no, "empty line"))
            continue
        try:
            obj = json.loads(line)
            validate_example(obj)
            good.append(obj)
        except json.JSONDecodeError as e:
            problems.append((line_no, f"invalid JSON: {e.msg}"))
        except ValueError as e:
            problems.append((line_no, str(e)))

print(f"valid examples: {len(good)}")
for line_no, msg in problems:
    print(f"  line {line_no}: {msg}")

valid examples: 2
  line 2: last message must be from the assistant
  line 3: invalid JSON: Expecting ',' delimiter
  line 4: empty line


In [44]:
# Application 5 : expert system whose RULES live in a JSON file (knowledge separated from code)
import operator
rules_file = BASE / "rules.json"
rules_file.write_text(json.dumps([
    {"name": "bearing failure", "all": [["temp", ">", 85], ["vibration", ">", 7.1]]},
    {"name": "cooling problem", "all": [["temp", ">", 85]]},
    {"name": "leak suspected",  "all": [["pressure", "<", 2.0]]},
], indent=2), encoding="utf-8")

OPS = {">": operator.gt, "<": operator.lt, ">=": operator.ge, "<=": operator.le, "==": operator.eq}

def load_rules(path):
    rules = []
    for r in json.loads(Path(path).read_text(encoding="utf-8")):
        conditions = [(field, OPS[op], value) for field, op, value in r["all"]]
        rules.append((r["name"], conditions))
    return rules

def diagnose(facts, rules):
    return [name for name, conds in rules
            if all(op(facts[field], value) for field, op, value in conds)] or ["normal"]

RULES = load_rules(rules_file)
print(diagnose({"temp": 92, "vibration": 8.0, "pressure": 3.0}, RULES))
print(diagnose({"temp": 70, "vibration": 3.0, "pressure": 1.5}, RULES))

['bearing failure', 'cooling problem']
['leak suspected']


In [45]:
# Application 6 : save a checkpoint safely + its hash , and verify it when loading
ckpt = BASE / "models" / "checkpoint.pkl"
payload = {"epoch": 3, "weights": [0.4, -1.2, 0.7], "val_loss": 0.62}

tmp = ckpt.with_suffix(".tmp")
with open(tmp, "wb") as f:
    pickle.dump(payload, f)
os.replace(tmp, ckpt)                                        # atomic
(ckpt.with_suffix(".sha256")).write_text(sha256_of(ckpt), encoding="utf-8")

def load_checkpoint(path):
    expected = path.with_suffix(".sha256").read_text(encoding="utf-8").strip()
    if sha256_of(path) != expected:
        raise ValueError(f"checkpoint {path.name} is corrupted (hash mismatch)")
    with open(path, "rb") as f:
        return pickle.load(f)

print(load_checkpoint(ckpt))
with open(ckpt, "ab") as f:
    f.write(b"!")                                             # simulate corruption
try:
    load_checkpoint(ckpt)
except ValueError as e:
    print("ValueError:", e)

{'epoch': 3, 'weights': [0.4, -1.2, 0.7], 'val_loss': 0.62}
ValueError: checkpoint checkpoint.pkl is corrupted (hash mismatch)


In [46]:
# summary : what is inside the demo folder now ?
for p in sorted(BASE.rglob("*")):
    if p.is_file() and "dataset" not in p.parts:
        print(f"{str(p.relative_to(BASE)):<38} {p.stat().st_size:>9,} bytes")

c.txt                                          0 bytes
config.json                                  194 bytes
data/manifest.csv                            480 bytes
data/processed/customers_clean.csv            86 bytes
data/raw/customers.csv                        90 bytes
data/raw/scores.tsv                           37 bytes
data/train.jsonl                             199 bytes
data/train_raw.jsonl                         292 bytes
for_excel.csv                                 37 bytes
hello.txt                                     30 bytes
image.png                                 10,248 bytes
important.txt                                  0 bytes
metrics.csv                                   91 bytes
models/best.json                              30 bytes
models/checkpoint.pkl                         88 bytes
models/checkpoint.sha256                      64 bytes
models/config.json                             4 bytes
models/epoch_1.pt                              4 bytes
models/epo

In [47]:
# cleanup (set to True to delete the demo folder)
CLEANUP = False
if CLEANUP:
    shutil.rmtree(BASE)
print("demo folder kept:", BASE.exists())

demo folder kept: True


## Exercises
1. write `count_words(path)` that returns the 5 most common words of a UTF-8 text file (lower-case , no punctuation).
2. what happens to `notes.txt` if you run the "w" cell twice ? and the "a" cell twice ? explain.
3. write a function that converts `customers.csv` to a JSON file (list of dicts) with correct number types.
4. write `latest_checkpoint(folder)` that returns the path of the checkpoint with the highest epoch , or `None` if there is none.
5. in Application 3 , what happens if you run the cell twice ? how would you avoid duplicated epochs ?
6. add a new rule to `rules.json` (without touching the python code) and run Application 5 again.
7. why must we NOT use `pickle.load` on a file downloaded from the internet ? what can we use instead for model weights ?
8. read `server.log` and write a new CSV with the number of INFO / WARNING / ERROR lines per module.